In [2]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL, ExtendedUnbinnedNLL
from scipy.stats import expon, norm, uniform, chi2
from collections.abc import Callable
import plotly.graph_objects as go
import inspect
import pandas as pd

In [18]:
def gauss_cdf(x, a, mu, sigma):
    return a * norm.cdf(x, mu, sigma)

def gauss_pdf (x, a, mu, sigma):
    return a * norm.pdf(x, mu, sigma)

In [34]:
data = pd.read_csv('/home/matti/uni/lab/Lab-muons/data_sim_exp_unif.txt', sep  = ' ')
# print(data)
data2 = pd.read_csv('/home/matti/uni/lab/Lab-muons/data_sim_exp_unif2.txt', sep = ' ')
data4 = pd.read_csv('/home/matti/uni/lab/Lab-muons/data_sim_exp_unif4.txt', sep = ' ')

# Tau

In [40]:
nbin = 10
count, edges = np.histogram(data['tau'], bins= nbin, density = False)
count2, edges2 = np.histogram(data2['tau'], bins = nbin, density = False)
count4, edges4 = np.histogram(data4['tau'], bins = 100, density = False)
fig = go.Figure()
fig.add_trace(go.Bar(x = edges, y = count, width = np.diff(edges), name ='tau distribution'))
fig.add_trace(go.Bar(x = edges2, y = count2, width = np.diff(edges2),))
fig.add_trace(go.Bar(x = edges4, y = count4, width = np.diff(edges4)))
fig.update_layout(xaxis_title = "Time [s]", yaxis_title = "Counts")
fig.show()

In [20]:
cost = ExtendedBinnedNLL(count, edges, gauss_cdf)
m = Minuit(cost, 200, 2.085e-6, 0.3e-8)
m.fixed['a'] = True
m.migrad()

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3.883 (χ²/ndof = 0.5)      │              Nfcn = 79               │
│ EDM = 4.05e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a     │    200    │     2     │            │            │         │         │  yes  │
│ 1 │ mu    │ 2.0904e-6 │ 0.0020e-6 │            │            │         │         │       │
│ 2 │ sigma │  27.6e-9  │  1.6e-9   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬────────────────────────────┐
│       │        a       mu    sigma │
├───────┼────────────────────────────┤
│     a │        0    0e-18        0 │
│    mu │    0e-18 4.15e-18        0 │
│ sigma │        0        0 2.67e-18 │
└───────┴────────────────────────────┘

In [7]:
print(m.values[1])
print("distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura):", 
      abs((m.values[1] - 2.1193010893993915e-06))/np.sqrt((m.errors[1])**2 + (5.37218123526306e-08)**2) )

2.09036407663405e-06
distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura): 0.5382585999563051


In [45]:
cost = ExtendedBinnedNLL(count4, edges4, gauss_cdf)
m = Minuit(cost, 1e5, 2.09e-6, 0.1e-6)
m.fixed['a'] = True
m.migrad()
# m.minos()
# m.draw_mnprofile('mu')

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 141.4 (χ²/ndof = 1.4)      │              Nfcn = 72               │
│ EDM = 2.84e-08 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a     │   100e3   │    1e3    │            │            │         │         │  yes  │
│ 1 │ mu    │2.09019e-6 │0.00004e-6 │            │            │         │         │       │
│ 2 │ sigma │ 12.451e-9 │ 0.028e-9  │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬────────────────────────────┐
│       │        a       mu    sigma │
├───────┼────────────────────────────┤
│     a │        0        0        0 │
│    mu │        0 1.55e-21        0 │
│ sigma │        0        0 7.77e-22 │
└───────┴────────────────────────────┘

# a

In [8]:
count, edges = np.histogram(data["a"], bins = 10)

fig = go.Figure()
fig.add_trace(go.Bar(x = edges, y = count, width = np.diff(edges), name ='tau distribution'))
fig.update_layout(xaxis_title = "Time [s]", yaxis_title = "Counts")
fig.show()

In [9]:
cost = ExtendedBinnedNLL(count, edges, gauss_cdf)
m = Minuit(cost, 200, 1, 5e-3)
m.fixed['a'] = True
m.migrad()

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 6.794 (χ²/ndof = 0.8)      │              Nfcn = 40               │
│ EDM = 3.63e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a     │    200    │     2     │            │            │         │         │  yes  │
│ 1 │ mu    │ 998.9e-3  │  0.5e-3   │            │            │         │         │       │
│ 2 │ sigma │  6.5e-3   │  0.4e-3   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬────────────────────────────┐
│       │        a       mu    sigma │
├───────┼────────────────────────────┤
│     a │        0        0        0 │
│    mu │        0 2.34e-07  0.02e-6 │
│ sigma │        0  0.02e-6 1.56e-07 │
└───────┴────────────────────────────┘

In [10]:
print(m.values[1])
print("distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura):", 
      abs((m.values[1] - 1.385))/np.sqrt((m.errors[1])**2 + (0.024)**2) )

0.9988938771969841
distanza sigma da valore trovato con questa interpolazione e interpolazione dataset (errore somma in quadratura): 16.08449200550579
